# Fingerprints for the MEVID-SC TRAIN split (CSCI + the OSNet baseline)

Needed for the learned light verifier (trained on the 104 train people, who never appear in test) and for a larger calibration pool.

- **CSCI:** runs the authors' released MEVID checkpoint, unchanged, through its own evaluation code, with the train clips presented as the evaluation set, and saves one fingerprint per train clip. Same patches as the test run.
- **OSNet baseline (OSNet):** same weights, loader and per-clip template as the OSNet baseline's identity layer.

Needs an **A100** runtime. About 10 min to unpack the 17.4 GB archive, about 45 to 60 min for CSCI, about 10 min for OSNet. Everything is saved to Drive; rerunning skips finished parts.

In [ ]:
# 1. Drive, paths, unpack the MEVID-SC train archive to local disk
import os, glob, subprocess, sys, numpy as np
from google.colab import drive
drive.mount('/content/drive')

SD = '/content/drive/Shareddrives/*'
def find(pats):
    for p in pats:
        h = sorted(glob.glob(p, recursive=True))
        if h:
            return os.path.dirname(h[0])
    raise FileNotFoundError(pats[0])

SC_DIR = find([f'{SD}/Data/MEVID-SameClothes/SUBSET_REPORT.json',
               f'{SD}/**/MEVID-SameClothes/SUBSET_REPORT.json'])
CSCI_OUT = find([f'{SD}/Data/MEVID-results/CSCI/'
                 'ez_eva02_vid_hybrid_extra_best.pth',
                 f'{SD}/**/CSCI/ez_eva02_vid_hybrid_extra_best.pth'])
RES = os.path.dirname(CSCI_OUT)
CS_SC = f'{RES}/CSCI-SameClothes'
TR_SC = f'{RES}/Tracker-SameClothes'
for d in (CS_SC, TR_SC):
    os.makedirs(d, exist_ok=True)

LOCAL = '/content/sc_train'
if not os.path.isdir(f'{LOCAL}/bbox_train'):
    os.makedirs(LOCAL, exist_ok=True)
    print('Unpacking mevid-sc-bbox-train.tar (about 10 minutes) ...')
    subprocess.run(['tar', '-xf', f'{SC_DIR}/mevid-sc-bbox-train.tar',
                    '-C', LOCAL], check=True)
A = f'{SC_DIR}/annotations'
names = [l.strip() for l in open(f'{A}/train_name.txt') if l.strip()]
info = np.loadtxt(f'{A}/track_train_info.txt').astype(np.int64)
n_img = sum(len(f) for _, _, f in os.walk(f'{LOCAL}/bbox_train'))
print(f'Train clips: {len(info)} | people: {len(np.unique(info[:, 2]))} '
      f'| images listed: {len(names):,} | on disk: {n_img:,}')
assert n_img >= len(names), 'some train images are missing'


In [ ]:
# 2. Present the train clips to CSCI as its evaluation set
#    (CSCI only dumps features for the split it evaluates)
ROOT = '/content/mevid_sc_train_root'
os.makedirs(ROOT, exist_ok=True)
links = {'test_name.txt': f'{A}/train_name.txt',
         'track_test_info.txt': f'{A}/track_train_info.txt',
         'train_name.txt': f'{A}/train_name.txt',
         'track_train_info.txt': f'{A}/track_train_info.txt',
         'bbox_test': f'{LOCAL}/bbox_train'}
for k, v in links.items():
    if not os.path.exists(f'{ROOT}/{k}'):
        os.symlink(v, f'{ROOT}/{k}')

# queries: one clip per person that has the same person on another camera
# (only needed so CSCI's evaluation runs; every clip gets a feature)
pid_t, cam_t = info[:, 2], info[:, 4]
q = []
for p in np.unique(pid_t):
    rows = np.flatnonzero(pid_t == p)
    for r in rows:
        if (cam_t[rows] != cam_t[r]).any():
            q.append(r)
            break
np.savetxt(f'{ROOT}/query_IDX.txt', np.array(q), fmt='%d')
print(f'{len(q)} queries, {len(info) - len(q)} gallery clips; root:', ROOT)


In [ ]:
# 3. CSCI code, weights (already on Drive) and the same two patches as
#    the test run: a feature dump, and skip loading a training clip
REPO = '/content/ICCV-CSCI-Person-ReID'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/ppriyank/'
                    'ICCV-CSCI-Person-ReID.git', REPO], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'timm==1.0.9', 'einops', 'yacs', 'mmengine', 'decord',
                'tensorboardX', 'tensorboard', 'termcolor', 'tabulate',
                'h5py', 'scikit-learn', 'lmdb', 'albumentations',
                'torchinfo', 'grad-cam', 'opencv-python-headless'],
               check=True)
WT = f'{CSCI_OUT}/ez_eva02_vid_hybrid_extra_best.pth'

f = f'{REPO}/processor/eval_mevid.py'; src = open(f).read()
anchor = '    distmat = distmat.cpu()\n'
hook = (anchor +
        "    if local_rank == 0 and os.environ.get('CSCI_DUMP'):\n"
        "        import numpy as _np\n"
        "        _np.savez(os.environ['CSCI_DUMP'], "
        "qf=qf.float().cpu().numpy(), gf=gf.float().cpu().numpy(), "
        "q_pids=q_pids.numpy(), g_pids=g_pids.numpy(), "
        "q_camids=q_camids.numpy(), g_camids=g_camids.numpy())\n")
if 'CSCI_DUMP' not in src:
    assert src.count(anchor) == 1, 'eval_mevid.py changed upstream'
    src = src.replace(anchor, hook)
    if '\nimport os' not in src:
        src = 'import os\n' + src
    open(f, 'w').write(src)
f = f'{REPO}/data/dataset_loader.py'; src = open(f).read()
old_l = ('        self.return_index = return_index\n'
         '        self.__getitem__(0)\n')
new_l = ('        self.return_index = return_index\n'
         '        if not train:\n            self.__getitem__(0)\n')
if old_l in src:
    open(f, 'w').write(src.replace(old_l, new_l))
print('CSCI ready; weights:', WT)


In [ ]:
# 4. Run CSCI on the train clips and save one fingerprint per clip
DUMP = f'{CS_SC}/csci_sc_train_dump.npz'
OUTF = f'{CS_SC}/train_templates_csci_sc.npz'
if not os.path.exists(DUMP):
    env = dict(os.environ, CSCI_DUMP=DUMP, PYTHONPATH=REPO)
    cmd = ['torchrun', '--nproc_per_node=1', '--master_port=12357',
           'train_two_step.py', '--eval', '--resume',
           '--config_file', 'configs/mevid_eva02_l_cloth.yml',
           'DATA.ROOT', ROOT + '/', 'TEST.WEIGHT', WT,
           'SOLVER.SEED', '12345', 'TRAIN.TRAIN_VIDEO', 'True',
           'DATA.DATASET', 'mevid', 'MODEL.NAME',
           'ez_eva02_vid_hybrid_extra', 'TRAIN.COLOR_PROFILE', '17',
           'DATA.TEST_BATCH', '32']
    with open(f'{CS_SC}/csci_sc_train_log.txt', 'w') as log:
        p = subprocess.Popen(cmd, cwd=REPO, env=env, text=True,
                             stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT)
        for line in p.stdout:
            log.write(line)
            if any(k in line for k in ['top1', 'mAP', 'Error',
                                       'Traceback', 'Extract']):
                print(line.rstrip())
        p.wait()
    print('Exit code:', p.returncode, '(a non-zero code after the dump is '
          'CSCI\'s optional location/scale protocol; it is harmless)')
assert os.path.exists(DUMP), 'CSCI stopped before dumping; see the log'

z = np.load(DUMP)
qidx = np.loadtxt(f'{ROOT}/query_IDX.txt').astype(int)
gidx = np.setdiff1d(np.arange(len(info)), qidx)
assert (z['q_pids'] == info[qidx, 2]).all(), 'query order mismatch'
assert (z['g_pids'] == info[gidx, 2]).all(), 'gallery order mismatch'
T = np.zeros((len(info), z['qf'].shape[1]), np.float32)
T[qidx], T[gidx] = z['qf'], z['gf']
T /= np.maximum(np.linalg.norm(T, axis=1, keepdims=True), 1e-8)
np.savez(OUTF, templates=T, pid=info[:, 2], oid=info[:, 3],
         cam=info[:, 4])
print('Saved', OUTF, T.shape)


In [ ]:
# 5. The OSNet baseline's OSNet on the train clips (same weights, loader and
#    per-clip template as the OSNet baseline's identity layer)
OUTT = f'{TR_SC}/train_templates_tracker_sc.npz'
if os.path.exists(OUTT):
    print('Already saved:', OUTT)
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torchreid', 'gdown'], check=True)
    import torch, torch.nn.functional as F, torchreid, gdown
    import torchvision.transforms as TV
    from PIL import Image
    from torch.utils.data import Dataset, DataLoader
    from tqdm.auto import tqdm
    model = torchreid.models.build_model(name='osnet_x1_0',
                                         num_classes=1000, loss='softmax',
                                         pretrained=True)
    ckpt = os.path.expanduser('~/.cache/torch/checkpoints/'
                              'osnet_x1_0_msmt17.pth')
    os.makedirs(os.path.dirname(ckpt), exist_ok=True)
    for url in ['https://drive.google.com/uc?id=112EMUfBPYeYg70w-'
                'syK6V6Mx8-Qb9Q1M', 'https://drive.google.com/uc?id='
                '1IosIFlLiulGIjwW3H8uMRmx3MzPwf86x']:
        if not os.path.exists(ckpt):
            try:
                gdown.download(url, ckpt, quiet=True, fuzzy=True)
            except Exception as e:
                print('download failed:', e)
    torchreid.utils.load_pretrained_weights(model, ckpt)
    model = model.cuda().eval()
    TF = TV.Compose([TV.Resize((256, 128)), TV.ToTensor(),
                     TV.Normalize([0.485, 0.456, 0.406],
                                  [0.229, 0.224, 0.225])])
    n2p = {f: os.path.join(dp, f) for dp, _, fs in
           os.walk(f'{LOCAL}/bbox_train') for f in fs}
    items = [(n2p[os.path.basename(names[k])], t)
             for t, (s, e) in enumerate(info[:, :2])
             for k in range(s, e + 1)]

    class Frames(Dataset):
        def __len__(self): return len(items)
        def __getitem__(self, i):
            p, t = items[i]
            return TF(Image.open(p).convert('RGB')), t

    per = [[] for _ in range(len(info))]
    dl = DataLoader(Frames(), batch_size=512, pin_memory=True,
                    num_workers=min(12, os.cpu_count() or 2))
    with torch.no_grad():
        for x, t in tqdm(dl, desc=f'OSNet on {len(items):,} frames'):
            f = model(x.cuda(non_blocking=True))
            f = F.normalize(f[0] if isinstance(f, (tuple, list)) else f,
                            dim=1).cpu().numpy()
            for k, ti in enumerate(t.numpy()):
                per[ti].append(f[k].astype(np.float16))
    tm = np.zeros((len(info), 512), np.float32)
    sm = np.ones(len(info), np.float32)
    for t in range(len(info)):
        Ft = np.stack(per[t]).astype(np.float32)
        m = Ft.mean(0); m /= max(np.linalg.norm(m), 1e-8)
        tm[t], sm[t] = m, float((Ft @ m).min())
    np.savez(OUTT, zs=tm, zs_smin=sm, pid=info[:, 2], oid=info[:, 3],
             cam=info[:, 4])
    print('Saved', OUTT, tm.shape)
